# 第44课 · 给声音装上时间轴——亲手写 STFT（分帧 + 加窗 + FFT），与 aurora 逐点对齐

**目标**：从零实现完整的 STFT，使输出与 `aurora.audio.stft.stft()` 在数值上严格一致（`atol=1e-9`）。

🔗 **Aurora 连接**
- 重写目标：`src/aurora/audio/stft.py` — `stft()`, `frame_signal()`
- 测试基准：`tests/audio/test_transforms.py` — 确认 aurora FFT 与 numpy 误差 < 1e-9


← **上一课**　[L43 · STFT 原理](L43_stft.ipynb)

> 上节课学习了 **STFT 原理**：短时傅里叶变换：给信号加时间戳，时频分辨率 tradeoff。  
> 本课将探讨 **亲手写 STFT**。

## 本课剧情：为什么音乐软件能显示频谱图？

打开 Audacity 或 Ableton，点一下"频谱图视图"，你立刻看到音乐随时间变化的频率成分——低音鼓在底部闪光，人声在中频律动，高频镲片上方点点起伏。

这张图叫**声谱图（spectrogram）**，背后的算法就是 STFT。流程只有三步：

```
1. frame_signal(x, win_len, hop)   → 把信号切成帧，shape (n_frames, win_len)
2. frames *= window                → 每帧乘 Hann 窗，消除边缘跳变
3. STFT = fft(frames, axis=1)      → 对每帧做 FFT，shape (n_frames, win_len//2+1)
```

第一步 L43 已实现，第二步 L41 已掌握，本课把三步串成一条管线：`my_stft(x, win_len, hop)`。

**关键参数的直觉**：

| 参数 | 含义 | 影响 |
|---|---|---|
| `win_len` | 帧长（采样点） | 越大 → 频率越细（Δf=sr/win） |
| `hop` | 帧步进 | 越小 → 时间分辨率越高（帧数越多） |
| `window` | 窗函数 | Hann：旁瓣低 31 dB；rect：泄漏 |

本节目标：实现 `my_stft`，与 `aurora.audio.stft.stft` 误差 < 1e-10。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, chirp


## 概念 1：STFT 流水线

STFT 的核心公式：

```
STFT[t, f] = sum_{n=0}^{N-1}  x[t*hop + n] * w[n] * exp(-j*2*pi*f*n/N)
```

其中 `N` 是窗长（`win_len`），`w[n]` 是窗函数，`t` 是帧下标，`f` 是频率 bin 下标。 实现上等价于：先切帧 → 每帧乘窗 → 对每帧做长度 N 的 FFT → 取前 `N//2+1` 列。

`frame_signal(x, frame_length, hop_length)` 用 stride trick 零拷贝切帧：

```
idx[t, n] = t * hop + n      (t in [0, n_frames), n in [0, N))
frames = x[idx]               # shape: (n_frames, N)
```


In [ ]:
# 演示：切帧
from aurora.audio.stft import frame_signal

sr = 16000
x = sine(440, 0.5, sample_rate=sr)          # 0.5 秒 440 Hz 正弦波
win_len, hop = 2048, 512

frames = frame_signal(x, win_len, hop, center=True)
print(f"信号长度: {len(x)} 样本")
print(f"帧矩阵 shape: {frames.shape}")      # (n_frames, win_len)
print(f"n_frames = 1 + (padded_len - win_len) // hop")


## 概念 2：复数输出与幅度谱

FFT 对实信号输出 `N` 个复数，但正负频率共轭对称，有效信息只有前 `N//2+1` 个 bin（DC 到 Nyquist）。

```
spectrum = np.fft.fft(windowed_frame)        # shape: (N,)  复数
rfft_out = spectrum[:N//2+1]                 # shape: (N//2+1,)  只保留正频率
```

STFT 把所有帧堆叠后形状为 `(n_frames, N//2+1)`，dtype 是 `complex128`。 幅度谱 `magnitude = np.abs(stft_out)` 消去相位，得到每个时频格的强度，shape 不变。


In [ ]:
# 演示：单帧 FFT 与复数输出
from aurora.audio.windows import get_window

win = get_window("hann", win_len, periodic=True)   # Aurora 与 numpy 一致的周期 Hann 窗
frame0 = frames[0] * win                           # 第 0 帧乘窗

full_fft = np.fft.fft(frame0)                      # 完整 FFT，shape (win_len,)
rfft_out = full_fft[:win_len // 2 + 1]            # 截取正频率部分

print(f"full_fft shape: {full_fft.shape}")
print(f"rfft_out shape: {rfft_out.shape}")         # (1025,) for win_len=2048
print(f"rfft_out dtype: {rfft_out.dtype}")         # complex128
print(f"magnitude[0:5]: {np.abs(rfft_out[:5])}")


## 概念 3：center=True 模式 — 让第 0 帧以 x[0] 为中心

默认 `center=False` 时，第 0 帧从 `x[0]` 开始，窗中心在 `x[win_len//2]`，信号开头的能量被压到很小。

`center=True` 在信号两端补 `win_len//2` 个样本（Aurora 用反射填充），使帧 `t` 的窗中心恰好落在 `x[t * hop]`：

```
pad = win_len // 2
x_padded = np.pad(x, pad, mode="reflect")
# 第 0 帧: x_padded[0 : win_len]  → 中心在 x_padded[pad] = x[0]
# 第 t 帧: x_padded[t*hop : t*hop + win_len]
```

反射填充（reflect padding）让首尾帧也能被完整分析，是 librosa / Aurora 的默认行为。它会在边界处把信号镜像过去，这样就不会像零填充（zero padding）那样，在边界上凭空多出一段幅度斜坡（boundary transient）。


In [ ]:
# 演示：center=True 补零效果
pad = win_len // 2
x_padded = np.pad(x, pad, mode="reflect")

print(f"原始信号长度: {len(x)}")
print(f"反射填充后长度: {len(x_padded)}")
print(f"第 0 帧中心位置（补零后下标）: {pad}  → 对应 x[0]")

# 不补零 vs 补零：第 0 帧窗中心的幅度对比
frames_no_center = frame_signal(x, win_len, hop, center=False)
frames_center    = frame_signal(x, win_len, hop, center=True)

mag_no = np.abs(np.fft.fft(frames_no_center[0] * win)[:win_len//2+1])
mag_ctr = np.abs(np.fft.fft(frames_center[0]    * win)[:win_len//2+1])

print(f"center=False 第0帧峰值: {mag_no.max():.4f}")
print(f"center=True  第0帧峰值: {mag_ctr.max():.4f}")
print("注：center=True 的第 0 帧左半是反射镜像，两半相位相反、能量部分抵消，峰值反而更低。")
print("   center=True 的意义不在能量大小，而在时间对齐：第 t 帧窗中心严格对准 x[t*hop]。")


## 1. ✏️ 实现 `my_stft(x, win_len=2048, hop=512, window="hann")`

**三步串联**：

| 步骤 | 代码 | 说明 |
|---|---|---|
| 1 | `frames = frame_signal(x, win_len, hop)` | 切帧，shape (n_frames, win_len) |
| 2 | `frames *= get_window(window, win_len)` | 乘窗，每帧同一个窗向量 |
| 3 | 逐帧 `aurora_fft(f)[:win_len//2+1]` 再 `np.stack` | 铁律：引擎用 L37-39 手写 FFT（win_len=2048 是 2 的幂）；`np.fft` 仅作参考对照 |

**验收标准**：
- 输出 shape = `(n_frames, win_len//2+1)`（单边频谱）
- `np.allclose(my_stft(x, win_len, hop), aurora_stft(x, win_len, hop), atol=1e-10)`
- `win_len=2048, hop=512, sr=16000, duration=0.5`（8000 样本）→ aurora 默认 `center=True`，n_frames = 1 + 8000//512 = 16（若 `center=False` 才是 1+(8000-2048)//512 = 12）

**可用导入**：
```python
from aurora.audio.stft import frame_signal
from aurora.audio.windows import get_window
from aurora.audio.transforms import fft as aurora_fft  # 从零实现的 Cooley-Tukey FFT
```

## 常见失败模式（写 my_stft 前读一遍）

- 忘记 `rfft` / 只取 `win_len//2+1` 个 bin → shape 多一倍或对齐失败
- `center=True` 时帧数与手推公式差几帧 → **以 `aurora.stft.stft` 为准**
- 帧上未乘窗直接 FFT → 旁瓣泄漏爆炸（L36）
- `hop` 与 `win_len` 搞反 → n_frames 全错

卡住 → 回 L43 `frame_signal` / L41 加窗 / L39 FFT；完整参考见 `solutions/L44_stft_implement_solutions.md`。


## 半步练习 · 单帧 STFT（不替代 my_stft）

先只对**第 0 帧**跑通：`frame_signal` → `× hann` → `rfft` → 幅度。
与 `aurora.stft` 第 0 行对齐后，再写 `for` 循环拼完整 `my_stft`。


In [ ]:
import numpy as np
from aurora.audio.io import sine
from aurora.audio.stft import frame_signal, stft
from aurora.audio.windows import hann
from aurora.audio.transforms import rfft as aurora_rfft  # 手写 FFT 引擎（win_len=2048 是 2 的幂）

sr = 16_000
x = sine(440.0, 0.5, sample_rate=sr)
win_len, hop = 2048, 512
win = hann(win_len, periodic=True)

frames = frame_signal(x, win_len, hop, center=True)
row0 = frames[0] * win
spec0 = aurora_rfft(row0)   # 铁律：引擎走手写 FFT；np.fft 仅作对照
ref = stft(x, n_fft=win_len, hop_length=hop, window="hann")

np.testing.assert_allclose(spec0, ref[0], atol=1e-9)
print("✅ 第 0 帧 rfft 与 aurora.stft[0] 一致；可继续实现完整 my_stft")


In [ ]:
def my_stft(x, win_len=2048, hop=512, window="hann"):
    """手写 STFT，返回 complex128 数组 shape (n_frames, win_len//2+1)。"""
    from aurora.audio.windows import get_window
    from aurora.audio.stft import frame_signal
    from aurora.audio.transforms import fft as aurora_fft  # L37-39 手写 Cooley-Tukey FFT

    # ✏️ TODO 1: 生成窗函数 win，shape (win_len,)，periodic=True
    raise NotImplementedError("TODO: 用 aurora.audio.windows.hann(n_fft) 生成窗函数")

    # ✏️ TODO 2: 切帧，shape (n_frames, win_len)，center=True
    raise NotImplementedError("TODO: 分帧 shape=(n_frames, n_fft)")

    # ✏️ TODO 3: 每帧乘窗
    raise NotImplementedError("TODO: windowed = frames * win")

    # ✏️ TODO 4: 对每帧做 fft，取前 win_len//2+1 列
    n_bins = win_len // 2 + 1
    spectra = []
    for f in windowed:
        pass  # ✏️ TODO 4: append aurora_fft(f)[:n_bins] to spectra（铁律：引擎用手写 FFT；np.fft 仅作对照）

    # ✏️ TODO 5: stack 成 2D 数组返回
    return None


In [ ]:
from aurora.audio.stft import stft

sr = 16000
x = sine(440, 0.5, sample_rate=sr)

try:
    out = my_stft(x, win_len=2048, hop=512, window="hann")
except (NotImplementedError, TypeError):
    print("⬜ 请先完成上面的 my_stft 练习，再运行本格")
else:
    ref = stft(x, n_fft=2048, hop_length=512, window="hann")

    assert out is not None, "还没有实现，返回了 None"
    assert out.shape == ref.shape, f"shape 不符：got {out.shape}, expected {ref.shape}"
    assert out.dtype == np.complex128, f"dtype 应为 complex128，got {out.dtype}"
    assert np.allclose(out, ref, atol=1e-9), (
        f"数值不匹配，最大误差 {np.abs(out - ref).max():.2e}"
    )
    print(f"✅ shape {out.shape}，dtype {out.dtype}，max_err < 1e-9")


## 参数实验：`win_len` 对频谱图的影响

固定 `hop=256`，对比 `win_len = 128 / 512 / 2048` 三种设置：

| `win_len` | 频率分辨率 `sr/win_len` | 时间分辨率 `hop/sr` | 预期现象 |
|---|---|---|---|
| 128 | 125 Hz/bin | 16 ms | 时间精细，频率模糊 — 440 Hz 的正弦峰很宽 |
| 512 | 31 Hz/bin | 16 ms | 中等均衡 |
| 2048 | 7.8 Hz/bin | 16 ms | 频率锐利 — 440 Hz 的峰只占 1-2 bin |

`win_len` 越大，频率分辨率越高，但每帧时长也越长（时间分辨率下降）。 这是 STFT 的**时频不确定性**（Heisenberg-Gabor 限制）的直接体现。


In [ ]:
sr = 16000
x_chirp = chirp(200, 4000, 1.0, sample_rate=sr)   # 1 秒线性 chirp：200 → 4000 Hz
hop = 256

# ⚠️ 防呆：my_stft 尚未实现（抛 NotImplementedError 或返回 None）时跳过可视化
try:
    _guard = my_stft(x_chirp, win_len=512, hop=hop)
except (NotImplementedError, TypeError):
    _guard = None
if _guard is None:
    print("⚠️  my_stft 尚未实现，跳过可视化 — 完成练习后重新运行本格")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, wl in zip(axes, [128, 512, 2048]):
        S = np.abs(my_stft(x_chirp, win_len=wl, hop=hop)) ** 2
        S_db = 10 * np.log10(S.T + 1e-8)              # 转 dB，转置让频率在 y 轴

        im = ax.imshow(
            S_db,
            origin="lower",
            aspect="auto",
            extent=[0, len(x_chirp)/sr, 0, sr/2],
            vmin=-60, vmax=0,
            cmap="inferno",
        )
        ax.set_title(f"win_len={wl}  ({sr/wl:.0f} Hz/bin)")
        ax.set_xlabel("时间 (s)")
        ax.set_ylabel("频率 (Hz)")

    plt.colorbar(im, ax=axes, label="dB")
    plt.suptitle("STFT 参数实验：win_len 对时频分辨率的影响", fontsize=13)
    plt.tight_layout()
    plt.show()


## 🎯 未来的回报 (Future Payoff)
今天你亲手拼出的 `my_stft`（分帧 → 加窗 → FFT），会在 **L45 声谱图 / L47 log-Mel / L50 MFCC** 原封不动地充当第一级流水线——后面所有音频特征都从这张时频矩阵长出来。这一课调试 `center` 与加窗的痛苦，是在替后面五课省时间。

## 本课收束

`my_stft` 复现了 Aurora `stft()` 的完整计算图：`frame_signal` 切帧 → 乘 Hann 窗 → 手写 `aurora_fft`（`aurora.audio.transforms.fft`，L37-39 的 Cooley-Tukey）取前 `win_len//2+1` 列， 输出 `(n_frames, win_len//2+1)` 的 `complex128` 矩阵，数值误差 < 1e-9。 `np.abs()` 把复数矩阵压缩为幅度谱，`win_len` 直接控制频率分辨率（越大越细），与帧时长成正比反映时频不确定性。 下一课（L45）将在 STFT 复数矩阵上生成声谱图（spectrogram），用 pcolormesh 热力图展示时频能量分布，直观验证 `win_len` 对频率分辨率的影响。


## ✏️ 闭卷推导检查格 — STFT 帧数公式

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：给定：
- 信号长度 $L$（采样点数）
- 窗长 $N_{fft}$
- 跳步 $H$（hop_length）
- center 补零模式（在信号两端各补 $N_{fft}/2$ 个零）

推导 STFT 输出的帧数 $n_{frames}$ 的计算公式。

写出公式并说明每一步的物理含义。

（在此处写推导...）

In [ ]:
# 验证：公式计算结果与 aurora.audio.stft 输出帧数一致
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.stft import stft

SR, N_FFT, HOP = 16000, 1024, 256
signal = np.sin(2 * np.pi * 440 * np.arange(SR) / SR).astype(np.float32)

spec = stft(signal, n_fft=N_FFT, hop_length=HOP)
actual_frames = spec.shape[0]

# 公式：center=True → 补零后长度 L + N_FFT；n_frames = 1 + L // HOP
formula_frames = 1 + len(signal) // HOP
assert actual_frames == formula_frames, (
    f"公式给出 {formula_frames}，实际 {actual_frames}，检查你的推导"
)
print(f"✅ 帧数验证通过：n_frames = {actual_frames}（公式 = {formula_frames}）")

In [ ]:
# ✏️ 本课自评
l44_review = {
    "stft_three_steps":        None,  # 记住 frame→window→fft 三步流程？True/False
    "my_stft_implemented":     None,  # my_stft 实现并通过 atol=1e-10 对比？True/False
    "n_frames_formula":        None,  # 推导 center=True 时 n_frames=1+L//hop 闭卷通过？True/False
    "win_vs_hop_tradeoff":     None,  # 理解 win_len↑→频率细、hop↓→时间密的权衡？True/False
    "whiteboard_passed":       None,  # 白板挑战帧数公式推导完成？True/False
    "single_frame_half_step":  None,  # 单帧 STFT 半步与 aurora.stft[0] 对齐？True/False
}

unfilled = [k for k, v in l44_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l44_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L44 全部通关！进入 L45：把 STFT 画成声谱图')

---

→ **下一课**　[L45 · 声谱图（spectrogram）](L45_spectrogram.ipynb)

> 下节课将学习 **声谱图（spectrogram）**：从 |STFT|² 功率谱到 dB 热力图，给声音拍一张时频 X 光片。